# 区域销售分析

配合本仓库的笔记本编辑器与内核使用。用 `Ctrl/Cmd+Enter` 运行当前单元格，
用工具栏“运行全部”按顺序跑完；点状态栏的**源码**可以看到原始的 `.ipynb` JSON。

数据来自同目录的 `sales_clean.csv`，其中故意留了两类脏数据：少量缺失区域，
和若干被录成负数的销售额。下面先按列名汇总，再做一次修正对比。

In [2]:
import csv
import sys
from collections import defaultdict
from pathlib import Path

## 读取数据

内核的工作目录就是这个笔记本所在的目录，所以相对路径就是字面意思——
这一条与 Jupyter 自己的前端行为一致。

In [3]:
with Path('sales_clean.csv').open(newline='', encoding='utf-8-sig') as handle:
    rows = list(csv.DictReader(handle))

# `utf-8-sig` also strips a byte-order mark; without it the *first* column name
# would silently carry an invisible prefix and every lookup on it would fail.
len(rows), rows[0]['订单日期'], rows[0]['区域'], rows[0]['销售额']

(1200, '2024-02-02', '华北', '1057.52')

## 修正缺失与异常

缺失区域归到“未知”；负销售额按 `单价 × 数量` 重算，与 `sales_analysis.py` 的处理一致。

In [5]:
def money(value):
    try:
        return float(str(value).replace(',', '').strip() or 0)
    except ValueError:
        return 0.0

fixed_negative = 0
unknown_region = 0
for row in rows:
    amount = money(row['销售额'])
    if amount <= 0:
        amount = round(money(row['单价']) * money(row['数量']), 2)
        fixed_negative += 1
    row['金额'] = amount
    if not (row.get('区域') or '').strip():
        row['区域'] = '未知'
        unknown_region += 1

print('修正负销售额', fixed_negative, '行', file=sys.stderr)
print('缺失区域', unknown_region, '行', file=sys.stderr)
fixed_negative, unknown_region

修正负销售额 0 行
缺失区域 0 行


(0, 0)

In [7]:
by_region = defaultdict(float)
by_category = defaultdict(float)
for row in rows:
    by_region[row['区域']] += row['金额']
    by_category[row.get('品类') or '未知'] += row['金额']

regions = sorted(by_region.items(), key=lambda pair: pair[1], reverse=True)
total = sum(by_region.values())
[(name, round(value, 2)) for name, value in regions]

[('华东', 1593336.71),
 ('华南', 1210243.24),
 ('华北', 993849.01),
 ('西南', 777588.66),
 ('东北', 468923.57),
 ('未知', 110441.6)]

## 渲染成表格

`display(HTML(...))` 产生一个 `text/html` bundle。工作台会先过滤再渲染，所以它显示为
真正的表格——这也是 `pandas` 的 `DataFrame` 走的那条富输出路径。

In [8]:
from IPython.display import HTML, display

body = ''.join(
    f'<tr><td>{name}</td><td align="right">{value:,.2f}</td>'
    f'<td align="right">{value / total if total else 0:.1%}</td></tr>'
    for name, value in regions
)
display(HTML(
    '<table><thead><tr><th>区域</th><th>销售额</th><th>占比</th></tr></thead>'
    f'<tbody>{body}</tbody></table>'
))

区域,销售额,占比
华东,"1,593,336.71",30.9%
华南,"1,210,243.24",23.5%
华北,"993,849.01",19.3%
西南,"777,588.66",15.1%
东北,"468,923.57",9.1%
未知,"110,441.60",2.1%


In [9]:
# 一个 Markdown 单元格会被渲染；这段代码的最后一个表达式是 execute_result。
best = max(by_category.items(), key=lambda pair: pair[1]) if by_category else ('无', 0)
{'品类': best[0], '销售额': round(best[1], 2), '区域数': len(regions)}

{'品类': '手机数码', '销售额': 3317874.7, '区域数': 6}

## 输出与中断

下面三个单元格分别演示 stdout / stderr、错误与彩色 traceback、以及可中断的长任务。

In [10]:
print('这一行是 stdout')
print('这一行是 stderr', file=sys.stderr)
'最后一个表达式成为 execute_result'

这一行是 stdout


这一行是 stderr


'最后一个表达式成为 execute_result'

In [11]:
# 错误会作为 `error` 输出存进文件，并带着它的彩色 traceback。
raise ValueError('这是一条故意抛出的错误')

ValueError: 这是一条故意抛出的错误

In [9]:
# 运行它，然后点工具栏的中断按钮：本单元格中止，内核仍可继续使用。
import time
for _ in range(60):
    time.sleep(0.5)
print('不会执行到这里')

KeyboardInterrupt: 